# Feature 1.4 — Data Cleaning Exploration

**Dự án:** HitRadar Pro | **EPIC:** EPIC 1 — Data Foundation  
**Owner:** Đạt | **Notebook:** `3.NOTEBOOKS/3.3.lam_sach_python/01_feature_1_4_cleaning_exploration.ipynb`

---

## Mục tiêu Feature 1.4

| Mục tiêu | Chi tiết |
|----------|----------|
| Hiểu cấu trúc raw data | Xác nhận format artists, genres, release_date |
| Xác định data issues | tempo=0, time_signature=0, duration outliers |
| Chốt cleaning rules | Trước khi chạy `clean_raw_to_clean.py` |
| Scope | Raw → Clean layer, KHÔNG làm ML/model |

**Giới hạn:**
- Không train model
- Không parse dict_artists.json làm genre source
- Không sửa raw data
- Không tạo fake data

In [1]:
import os, sys, ast, re, json
import psycopg2
import psycopg2.extras
import pandas as pd

# ── Connection ──────────────────────────────────────────────────────────
DB_PARAMS = dict(
    host='localhost',
    port=5432,
    user='postgres',
    password=os.environ.get('PGPASSWORD', ''),
    dbname='hitradar',
)

conn = psycopg2.connect(**DB_PARAMS)
print('Connected to:', DB_PARAMS['dbname'], '@', DB_PARAMS['host'])

Connected to: hitradar @ localhost


## 1. Row counts — raw tables

In [2]:
cur = conn.cursor()
for tbl in ('raw.raw_tracks', 'raw.raw_artists', 'raw.raw_artist_json'):
    cur.execute(f'SELECT COUNT(*) FROM {tbl}')
    print(f'{tbl}: {cur.fetchone()[0]:,}')
cur.close()

raw.raw_tracks: 586,672
raw.raw_artists: 1,162,095


raw.raw_artist_json: 573,856


<div style="padding: 15px; border-radius: 8px; background: #e8f4f8; border-left: 5px solid #007bb5; color: #1a4a5e; font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif; margin: 15px 0; box-shadow: 0 2px 5px rgba(0,0,0,0.05);">
    <strong style="font-size: 1.1em; color: #005f8d;">🔍 Nhận xét & Đánh giá:</strong><br>
    <div style="margin-top: 8px; line-height: 1.5;">Số lượng bản ghi lớn ở cả 3 bảng raw, cho thấy quá trình thu thập (crawl) dữ liệu thô đã thực hiện thành công. Cần chú ý dung lượng lớn khi xử lý trong bộ nhớ.</div>
</div>

## 2. Sample raw.raw_tracks

In [3]:
df_tracks = pd.read_sql(
    'SELECT id, name, popularity, duration_ms, explicit, artists, id_artists, release_date,'
    ' tempo, time_signature FROM raw.raw_tracks LIMIT 10',
    conn
)
df_tracks

C:\Users\Admin\AppData\Local\Temp\ipykernel_14876\4094747032.py:1: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_tracks = pd.read_sql(


,id,name,popularity,duration_ms,explicit,artists,id_artists,release_date,tempo,time_signature
0,1WSo0oe305PBXNQOcz9Nn2,Tu Sei L'Unica Donna Per Me - 2005 Remaster,49,230733,0,['Alan Sorrenti'],['7sCYC6bDTexE400qiLy4oq'],1979,106.318,4
1,5UBq9iGT7lfpibWr7KKqKe,Girls Talk,49,209267,0,['Dave Edmunds'],['65Gh3BfK84aTIugiRCgLBA'],1979-06-09,128.843,4
2,1I4el8B1ZZKF3OGzmXDH9T,Bomber,49,220160,0,['Motörhead'],['1DFr97A9HnbV3SKTJFu62M'],1979-10-27,105.748,4
3,20VyHR0s9CdmSLM06I4Y48,Better Not Look Down,49,202427,0,['B.B. King'],['5xLSa7l4IV1gsQfhAMvl0U'],1979-01-01,101.839,4
4,1AWSemPzuGu4A9lVhSsFWJ,Frederick,49,184333,0,['Patti Smith'],['0vYkHhJ48Bs3jWcvZXvOrP'],1979,124.870,4
5,1dYimSgEq46lJ30MKQP9l6,Love Comes To Everyone - Remastered 2004,49,276213,0,['George Harrison'],['7FIoB5PHdrMZVC3q2HE5MS'],1979-02-20,107.168,4
6,0QSFBx4LlChEpPlzPLChNJ,Coisinha do Pai,49,174867,0,['Beth Carvalho'],['56TkPi7rpmU8jTpkcK7FY3'],1979-09-28,137.384,4
7,3u9g38eyqu5eJxU4Mhhzam,This Night Won't Last Forever,49,241267,0,['Michael Johnson'],['2XKBOnP3qXHP2FpzKplAYh'],1979-01-01,106.613,4
8,7rVvt8t21yAfgo4e4Cxj05,Tô Voltando,49,238200,0,['Simone'],['0sgV4klGs1Y1dgbBi28JlD'],1979-10-02,101.249,4
9,3F1MiFu5a1Y3Z9dQZhLTAo,Chuck E's in Love - 45 Version,49,209573,0,['Rickie Lee Jones'],['0dYkMe3wK29DulSa0uR8Rq'],1979,113.402,4


<div style="padding: 15px; border-radius: 8px; background: #e8f4f8; border-left: 5px solid #007bb5; color: #1a4a5e; font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif; margin: 15px 0; box-shadow: 0 2px 5px rgba(0,0,0,0.05);">
    <strong style="font-size: 1.1em; color: #005f8d;">🔍 Nhận xét & Đánh giá:</strong><br>
    <div style="margin-top: 8px; line-height: 1.5;">Dữ liệu track có chứa các cột list bị lưu dưới dạng string (như <code>artists</code>, <code>id_artists</code>). Cột <code>release_date</code> có định dạng chưa đồng nhất.</div>
</div>

## 3. Sample raw.raw_artists

In [4]:
df_artists = pd.read_sql(
    'SELECT id, name, genres, followers, popularity FROM raw.raw_artists LIMIT 10',
    conn
)
df_artists

C:\Users\Admin\AppData\Local\Temp\ipykernel_14876\325746014.py:1: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_artists = pd.read_sql(


,id,name,genres,followers,popularity
0,0DheY5irMjBUeLybbCUEZ2,Armid & Amir Zare Pashai feat. Sara Rouzbehani,[],0.0,0
1,0DlhY15l3wsrnlfGio2bjU,ปูนา ภาวิณี,[],5.0,0
2,0DmRESX2JknGPQyO15yxg7,Sadaa,[],0.0,0
3,0DmhnbHjm1qw6NCYPeZNgJ,Tra'gruda,[],0.0,0
4,0Dn11fWM7vHQ3rinvWEl4E,Ioannis Panoutsopoulos,[],2.0,0
5,0DotfDlYMGqkbzfBhcA5r6,Astral Affect,[],7.0,0
6,0DqP3bOCiC48L8SM9gK4W8,Yung Seed,[],1.0,0
7,0Drs3maQb99iRglyTuxizI,Wi'Ma,[],0.0,0
8,0DsPeAi1gxPPnYjgpiEGSR,lentboy,[],0.0,0
9,0DtvnTxgZ9K5YaPS5jdlQW,addworks,[],20.0,0


<div style="padding: 15px; border-radius: 8px; background: #e8f4f8; border-left: 5px solid #007bb5; color: #1a4a5e; font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif; margin: 15px 0; box-shadow: 0 2px 5px rgba(0,0,0,0.05);">
    <strong style="font-size: 1.1em; color: #005f8d;">🔍 Nhận xét & Đánh giá:</strong><br>
    <div style="margin-top: 8px; line-height: 1.5;">Tương tự tracks, cột <code>genres</code> của artist cũng đang bị lưu dưới dạng chuỗi string của list. Sẽ cần bóc tách (parse) bằng hàm <code>ast.literal_eval</code>.</div>
</div>

## 4. Sample raw.raw_artist_json

In [5]:
df_json = pd.read_sql(
    "SELECT artist_id, value_count, semantic_status, raw_values FROM raw.raw_artist_json WHERE value_count > 0 LIMIT 5",
    conn
)
df_json

C:\Users\Admin\AppData\Local\Temp\ipykernel_14876\2479634933.py:1: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_json = pd.read_sql(


,artist_id,value_count,semantic_status,raw_values
0,0syMDHmkYbx5dG8bOnZ60z,20,RELATED_ARTIST_GRAPH,"[2htg8Ya9Fbuy2zGKeL5q9i, 4XcCjDpADpJ4VBldrgcDb..."
1,2UHdlb01NKkTJWI6HR4Anb,20,RELATED_ARTIST_GRAPH,"[19gJjLsekMhmsPPPdepPqD, 2BK0stHHGdkx5vxXpnPRN..."
2,1r2lgySVpFN6pSuaHJPGV8,20,RELATED_ARTIST_GRAPH,"[59PxQQ1kv7UxyqCMJlB85d, 0gI0npmVgViDsNucipxwr..."
3,2W3hqPGyQyQZb1ENna4GnO,20,RELATED_ARTIST_GRAPH,"[1r979jUDNeZJWjPMfzZddD, 6rJCyRjfFqUQf8LUzvHPF..."
4,59PxQQ1kv7UxyqCMJlB85d,20,RELATED_ARTIST_GRAPH,"[1r2lgySVpFN6pSuaHJPGV8, 1r979jUDNeZJWjPMfzZdd..."


<div style="padding: 15px; border-radius: 8px; background: #e8f4f8; border-left: 5px solid #007bb5; color: #1a4a5e; font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif; margin: 15px 0; box-shadow: 0 2px 5px rgba(0,0,0,0.05);">
    <strong style="font-size: 1.1em; color: #005f8d;">🔍 Nhận xét & Đánh giá:</strong><br>
    <div style="margin-top: 8px; line-height: 1.5;">Bảng này chứa dữ liệu raw JSON trích xuất từ API. Có cấu trúc phức tạp và cần xử lý khéo léo để trích xuất các mối quan hệ (related artists) mà không làm rác database.</div>
</div>

## 5. Kiểm tra artists / id_artists list-string

In [6]:
cur = conn.cursor()
cur.execute('SELECT artists, id_artists FROM raw.raw_tracks LIMIT 2000')
rows = cur.fetchall()
cur.close()

parse_ok, parse_fail = 0, 0
max_artists, single_artist, multi_artists = 0, 0, 0

for artists_str, id_artists_str in rows:
    try:
        lst = ast.literal_eval(str(id_artists_str))
        if isinstance(lst, list):
            parse_ok += 1
            n = len(lst)
            max_artists = max(max_artists, n)
            if n == 1: single_artist += 1
            elif n > 1: multi_artists += 1
    except Exception:
        parse_fail += 1

print(f'Sample size: {len(rows)}')
print(f'Parse OK:    {parse_ok}')
print(f'Parse FAIL:  {parse_fail}')
print(f'Max artists per track: {max_artists}')
print(f'Single artist tracks: {single_artist}')
print(f'Multi-artist tracks:  {multi_artists}')

Sample size: 2000
Parse OK:    2000
Parse FAIL:  0
Max artists per track: 7
Single artist tracks: 1912
Multi-artist tracks:  88


<div style="padding: 15px; border-radius: 8px; background: #e8f4f8; border-left: 5px solid #007bb5; color: #1a4a5e; font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif; margin: 15px 0; box-shadow: 0 2px 5px rgba(0,0,0,0.05);">
    <strong style="font-size: 1.1em; color: #005f8d;">🔍 Nhận xét & Đánh giá:</strong><br>
    <div style="margin-top: 8px; line-height: 1.5;">Phần lớn dữ liệu parse thành công. Có một lượng đáng kể các track hợp tác (multi-artists), do đó việc chuẩn hóa thành bảng n-n (track_artists) là hoàn toàn hợp lý.</div>
</div>

## 6. Kiểm tra genres list-string

In [7]:
cur = conn.cursor()
cur.execute("SELECT genres FROM raw.raw_artists LIMIT 5000")
rows = cur.fetchall()
cur.close()

unique_genres, empty_count, parse_fail = set(), 0, 0
for (genres_str,) in rows:
    if not genres_str or str(genres_str).strip() == '[]':
        empty_count += 1
        continue
    try:
        lst = ast.literal_eval(str(genres_str))
        if isinstance(lst, list):
            unique_genres.update(g.strip() for g in lst if isinstance(g, str))
    except Exception:
        parse_fail += 1

print(f'Sample size:    {len(rows)}')
print(f'Empty genres:   {empty_count}')
print(f'Parse FAIL:     {parse_fail}')
print(f'Unique genres (sample): {len(unique_genres)}')
print(f'Examples: {list(sorted(unique_genres))[:10]}')

Sample size:    5000
Empty genres:   3156
Parse FAIL:     0
Unique genres (sample): 907
Examples: ['abstract hip hop', 'acid house', 'acid jazz', 'acid trance', 'acoustic pop', 'adoracao', 'adventista', 'african electronic', 'african percussion', 'afro dancehall']


<div style="padding: 15px; border-radius: 8px; background: #e8f4f8; border-left: 5px solid #007bb5; color: #1a4a5e; font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif; margin: 15px 0; box-shadow: 0 2px 5px rgba(0,0,0,0.05);">
    <strong style="font-size: 1.1em; color: #005f8d;">🔍 Nhận xét & Đánh giá:</strong><br>
    <div style="margin-top: 8px; line-height: 1.5;">Có nhiều nghệ sĩ không có genre (empty genres). Số lượng unique genres cũng rất đa dạng, cần lưu thành một bảng từ điển (dictionary) riêng biệt.</div>
</div>

## 7. Phân tích release_date — 3 formats

In [8]:
cur = conn.cursor()
cur.execute('SELECT release_date FROM raw.raw_tracks')
rows = cur.fetchall()
cur.close()

re_full  = re.compile(r'^\d{4}-\d{2}-\d{2}$')
re_month = re.compile(r'^\d{4}-\d{2}$')
re_year  = re.compile(r'^\d{4}$')

fmt_counts = {'YYYY-MM-DD': 0, 'YYYY-MM': 0, 'YYYY': 0, 'unknown': 0, 'null': 0}
for (rd,) in rows:
    if rd is None:
        fmt_counts['null'] += 1
    elif re_full.match(str(rd)):
        fmt_counts['YYYY-MM-DD'] += 1
    elif re_month.match(str(rd)):
        fmt_counts['YYYY-MM'] += 1
    elif re_year.match(str(rd)):
        fmt_counts['YYYY'] += 1
    else:
        fmt_counts['unknown'] += 1

total = len(rows)
for fmt, cnt in fmt_counts.items():
    print(f'{fmt:15s}: {cnt:8,}  ({cnt/total*100:.1f}%)')

YYYY-MM-DD     :  448,081  (76.4%)
YYYY-MM        :    2,102  (0.4%)
YYYY           :  136,489  (23.3%)
unknown        :        0  (0.0%)
null           :        0  (0.0%)


<div style="padding: 15px; border-radius: 8px; background: #e8f4f8; border-left: 5px solid #007bb5; color: #1a4a5e; font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif; margin: 15px 0; box-shadow: 0 2px 5px rgba(0,0,0,0.05);">
    <strong style="font-size: 1.1em; color: #005f8d;">🔍 Nhận xét & Đánh giá:</strong><br>
    <div style="margin-top: 8px; line-height: 1.5;">Cột <code>release_date</code> tồn tại 3 định dạng: ngày đầy đủ, chỉ có tháng, và chỉ có năm. Cần thêm một cột <code>release_precision</code> và chuẩn hóa tất cả về định dạng ngày (YYYY-MM-DD) chuẩn để DB dễ query.</div>
</div>

## 8. Kiểm tra tempo = 0

In [9]:
df_tempo = pd.read_sql(
    "SELECT COUNT(*) FILTER (WHERE tempo::float = 0) AS tempo_zero,"
    "       COUNT(*) FILTER (WHERE tempo IS NULL) AS tempo_null,"
    "       COUNT(*) AS total FROM raw.raw_tracks",
    conn
)
print(df_tempo.to_string(index=False))
print('\nSample tempo=0 rows:')
pd.read_sql("SELECT id, name, tempo FROM raw.raw_tracks WHERE tempo::float = 0 LIMIT 5", conn)

 tempo_zero  tempo_null  total
        328           0 586672

Sample tempo=0 rows:


C:\Users\Admin\AppData\Local\Temp\ipykernel_14876\2887559809.py:1: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_tempo = pd.read_sql(
C:\Users\Admin\AppData\Local\Temp\ipykernel_14876\2887559809.py:9: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  pd.read_sql("SELECT id, name, tempo FROM raw.raw_tracks WHERE tempo::float = 0 LIMIT 5", conn)


,id,name,tempo
0,3sAYxq1986j3ydqLv6jwUJ,"Serenade for Strings in E Major, Op. 22, B. 52...",0.0
1,5NR2UScCobzDg4zpbZYIel,Sultry Ambiance for 2021,0.0
2,035fB4qKVUc0GOW0oJcKMF,Clean White Noise,0.0
3,1RliEhshETEeAcCmA1lFHh,Sleepless Years,0.0
4,7dTyZl4cBRKnPBeq1fdrHY,White Noise,0.0


<div style="padding: 15px; border-radius: 8px; background: #e8f4f8; border-left: 5px solid #007bb5; color: #1a4a5e; font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif; margin: 15px 0; box-shadow: 0 2px 5px rgba(0,0,0,0.05);">
    <strong style="font-size: 1.1em; color: #005f8d;">🔍 Nhận xét & Đánh giá:</strong><br>
    <div style="margin-top: 8px; line-height: 1.5;">Phát hiện một số lượng nhỏ các bài hát có <code>tempo = 0</code>. Trong lý thuyết âm nhạc, nhịp độ = 0 là dữ liệu lỗi/không hợp lệ, vì vậy ta nên chuyển chúng thành <code>NULL</code> trong clean layer.</div>
</div>

## 9. Kiểm tra time_signature = 0

In [10]:
df_ts = pd.read_sql(
    "SELECT time_signature::text AS ts, COUNT(*) AS cnt FROM raw.raw_tracks"
    " GROUP BY time_signature ORDER BY cnt DESC",
    conn
)
print(df_ts.to_string(index=False))

ts    cnt
 4 503808
 3  64523
 5  11400
 1   6604
 0    337


C:\Users\Admin\AppData\Local\Temp\ipykernel_14876\2994623831.py:1: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_ts = pd.read_sql(


<div style="padding: 15px; border-radius: 8px; background: #e8f4f8; border-left: 5px solid #007bb5; color: #1a4a5e; font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif; margin: 15px 0; box-shadow: 0 2px 5px rgba(0,0,0,0.05);">
    <strong style="font-size: 1.1em; color: #005f8d;">🔍 Nhận xét & Đánh giá:</strong><br>
    <div style="margin-top: 8px; line-height: 1.5;">Tương tự tempo, nhịp (time signature) = 0 là bất thường. Đa số âm nhạc là nhịp 4 (4/4). Cần xử lý các giá trị 0 thành <code>NULL</code>.</div>
</div>

## 10. Kiểm tra duration outliers

In [11]:
df_dur = pd.read_sql(
    "SELECT COUNT(*) FILTER (WHERE duration_ms::int < 10000) AS short_tracks,"
    "       COUNT(*) FILTER (WHERE duration_ms::int > 3600000) AS long_tracks,"
    "       MIN(duration_ms::int) AS min_ms, MAX(duration_ms::int) AS max_ms,"
    "       ROUND(AVG(duration_ms::int)::numeric, 0) AS avg_ms"
    " FROM raw.raw_tracks",
    conn
)
print(df_dur.to_string(index=False))
print('\nShortest tracks:')
display(pd.read_sql("SELECT id, name, duration_ms FROM raw.raw_tracks ORDER BY duration_ms::int ASC LIMIT 5", conn))
print('\nLongest tracks:')
display(pd.read_sql("SELECT id, name, duration_ms FROM raw.raw_tracks ORDER BY duration_ms::int DESC LIMIT 5", conn))

C:\Users\Admin\AppData\Local\Temp\ipykernel_14876\4136651108.py:1: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_dur = pd.read_sql(


 short_tracks  long_tracks  min_ms  max_ms   avg_ms
           26           83    3344 5621218 230051.0

Shortest tracks:


C:\Users\Admin\AppData\Local\Temp\ipykernel_14876\4136651108.py:11: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  display(pd.read_sql("SELECT id, name, duration_ms FROM raw.raw_tracks ORDER BY duration_ms::int ASC LIMIT 5", conn))


,id,name,duration_ms
0,4WeyR22Ax2fF9dY0NxgjFV,Pause Track,3344
1,52qf3kN9pExTlHdSlh3ZeR,Pause Track,3344
2,2s6e7KLoQ5hie3Cnh73v2v,Pause Track,3344
3,4SjlyAejCNUB4MrGM1KuVp,Pause Track,3344
4,35GlCW5aqb8iJAdLuUf7tF,Pause Track,4000



Longest tracks:


C:\Users\Admin\AppData\Local\Temp\ipykernel_14876\4136651108.py:13: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  display(pd.read_sql("SELECT id, name, duration_ms FROM raw.raw_tracks ORDER BY duration_ms::int DESC LIMIT 5", conn))


,id,name,duration_ms
0,3EEv9UCeZdn4MVFv8tsO1E,โครงสร้างแห่งสิ่งที่เรียกว่าชีวิต,5621218
1,7foc25ig7dibxvULPU2kBG,Brown Noise - 90 Minutes,5403500
2,6rGikpwOv3LXaHWVCYbMNC,New Year's Eve 2015 Party Hits - Full DJ Party...,5042185
3,7jTxNjSwPcPjSbK8829Vno,Surah Al-Araf,4995083
4,7r86YmJo79FRcAHuVeKZp8,Tech House The Yearbook 2018 - Continuous Mix 2,4864333


<div style="padding: 15px; border-radius: 8px; background: #e8f4f8; border-left: 5px solid #007bb5; color: #1a4a5e; font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif; margin: 15px 0; box-shadow: 0 2px 5px rgba(0,0,0,0.05);">
    <strong style="font-size: 1.1em; color: #005f8d;">🔍 Nhận xét & Đánh giá:</strong><br>
    <div style="margin-top: 8px; line-height: 1.5;">Có sự chênh lệch rất lớn về độ dài (duration): có track siêu ngắn (vài giây) và siêu dài (hàng giờ). Đây có thể là tiếng ồn hoặc podcast, tuy nhiên ở bước clean tạm thời chỉ cần ghi nhận log, không drop.</div>
</div>

## 11. Cleaning Rules — Chốt

Dựa trên phân tích trên, các rules áp dụng trong `clean_raw_to_clean.py`:

| Rule | Hành động |
|------|----------|
| **Missing values** | Giữ NULL — không drop row |
| **tempo = 0** | → `NULL` (DDL CHECK: tempo > 0) |
| **time_signature = 0** | → `NULL` (DDL CHECK: 1–5) |
| **duration outliers** | Giữ lại, ghi cảnh báo trong log |
| **release_date YYYY-MM-DD** | precision=`day`, giữ nguyên |
| **release_date YYYY-MM** | precision=`month`, normalize → YYYY-MM-01 |
| **release_date YYYY** | precision=`year`, normalize → YYYY-01-01 |
| **explicit 0/1** | → `boolean` |
| **artists/id_artists** | `ast.literal_eval` → `clean.track_artists` |
| **genres** | `ast.literal_eval` → `clean.genres` + `clean.artist_genres` |
| **dict_artists.json** | → `clean.artist_relations` ONLY (không phải genre source) |
| **FK filter** | `track_artists` và `artist_relations` chỉ insert khi artist đã có trong `clean.artists` |

In [12]:
conn.close()
print('Connection closed. Exploration complete.')
print('Next: run clean_raw_to_clean.py')

Connection closed. Exploration complete.
Next: run clean_raw_to_clean.py
